In [1]:
import pandas as pd
import yfinance as yf
import FinanceDataReader as fdr
import pyupbit
from datetime import datetime
from sqlalchemy import create_engine
import pymysql
import warnings
warnings.filterwarnings('ignore')

# 0. MySQL 연결 엔진 세팅
db_user = 'root'
db_pw = 'jang1003'
db_host = 'localhost'
db_port = '3306'
db_name = 'quant_db'

db_connection_str = f'mysql+pymysql://{db_user}:{db_pw}@{db_host}:{db_port}/{db_name}'
engine = create_engine(db_connection_str)

# 1. 가격 데이터 추출 (업비트 실시간 가격 적용)
def fetch_current_prices():
    # 달러 환율
    exchange_rate = yf.Ticker("USDKRW=X").history(period="1d")['Close'].iloc[-1]

    # 자산별 주가 세팅
    prices = {
        # 비트코인, 이더리움은 업비트에서 실시간 원화(KRW) 가격 직수입
        'BTC': pyupbit.get_current_price("KRW-BTC"),
        'ETH': pyupbit.get_current_price("KRW-ETH"),
        # 미국 주식은 yfinance에서 달러 가격을 가져와 환율 곱함
        'VOO': yf.Ticker("VOO").history(period="1d")['Close'].iloc[-1] * exchange_rate,
        'GOOGL': yf.Ticker("GOOGL").history(period="1d")['Close'].iloc[-1] * exchange_rate,
        'USD': exchange_rate,
        'KRW': 1.0
    }
    return prices, exchange_rate

# 2. 데이터 변환 및 리밸런싱 로직
def process_portfolio_data(input_csv):
    df = pd.read_csv(input_csv)
    prices, exchange_rate = fetch_current_prices()

    # 현재 가치 및 비중 계산
    df['Current_Price'] = df['Asset'].map(prices)
    df['Total_Value_KRW'] = df['Quantity'] * df['Current_Price']

    total_asset_value = df['Total_Value_KRW'].sum()
    df['Current_Weight_%'] = (df['Total_Value_KRW'] / total_asset_value) * 100
    df['Weight_Gap_%'] = df['Target_Weight'] - df['Current_Weight_%']

    actions, amt_krw, amt_usd, action_qtys = [], [], [], []
    usd_assets = ['VOO', 'GOOGL']

    for _, row in df.iterrows():
        asset = row['Asset']
        target_w = row['Target_Weight']
        current_w = row['Current_Weight_%']
        current_price = row['Current_Price']

        # 목표 비중으로 되돌리기 위한 필요 금액
        required_krw = total_asset_value * (row['Weight_Gap_%'] / 100)

        # 현금(USD, KRW)은 매매 신호에서 제외
        if asset in ['USD', 'KRW']:
            exec_krw = 0
            actions.append("CASH_BUFFER")
        else:
            # ±20% 밴드 전략 적용 (평단가 방어 로직 제거, 순수 비중 추종)
            buy_threshold = target_w * 0.8
            sell_threshold = target_w * 1.2

            if current_w <= buy_threshold:
                exec_krw = required_krw
                actions.append("BUY")
            elif current_w >= sell_threshold:
                # 손실 여부와 상관없이 비중이 넘치면 기계적 익절/손절(SELL) 수행
                exec_krw = abs(required_krw)
                actions.append("SELL")
            else:
                exec_krw = 0
                actions.append("HOLD")

        amt_krw.append(exec_krw)

        # 달러 환산 (미국 주식 전용)
        if asset in usd_assets and exec_krw > 0:
            amt_usd.append(exec_krw / exchange_rate)
        else:
            amt_usd.append(0)

        # 권장 수량
        action_qtys.append(exec_krw / current_price if exec_krw > 0 else 0)

    # 데이터 정리
    df['Action_Signal'] = actions
    df['Action_Amount_KRW'] = amt_krw
    df['Action_Amount_USD'] = amt_usd
    df['Action_Quantity'] = action_qtys
    df['Last_Updated'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    # DB 적재 및 CSV 저장
    df.to_sql(name='portfolio_log', con=engine, if_exists='append', index=False)
    df.to_csv("portfolio_for_tableau.csv", index=False)

    print(f"총 자산 가치: {total_asset_value:,.0f} 원")
    print("-" * 115)
    print(f"{'Asset':<10} | {'목표%':>5} | {'현재%':>5} | {'상태':<15} | {'주문금액(달러/원)':>18} | {'권장수량'}")
    print("-" * 115)

    for _, row in df.iterrows():
        asset = row['Asset']
        a_krw, a_usd, qty = row['Action_Amount_KRW'], row['Action_Amount_USD'], row['Action_Quantity']

        amt_str = f"${a_usd:,.2f}" if asset in usd_assets and a_usd > 0 else (f"{a_krw:,.0f}원" if a_krw > 0 else "-")
        qty_str = f"{qty:.4f} 개" if qty > 0 else "-"

        print(f"{asset:<10} | {row['Target_Weight']:>6.1f} | {row['Current_Weight_%']:>6.1f} | {row['Action_Signal']:<15} | {amt_str:>20} | {qty_str}")

process_portfolio_data('my_assets.csv')

총 자산 가치: 31,656,860 원
-------------------------------------------------------------------------------------------------------------------
Asset      |   목표% |   현재% | 상태              |         주문금액(달러/원) | 권장수량
-------------------------------------------------------------------------------------------------------------------
BTC        |   50.0 |   49.8 | HOLD            |                    - | -
VOO        |   20.0 |   20.0 | HOLD            |                    - | -
ETH        |   10.0 |   10.9 | HOLD            |                    - | -
GOOGL      |   10.0 |   10.0 | HOLD            |                    - | -
USD        |   10.0 |    9.2 | CASH_BUFFER     |                    - | -
